# Neurotransmitter Probability Variance across Drosophila Neuropils

This projects aims to answer - how do neurotransmitter probability distributions vary across neuropils in Drosophila?
The datasets should be downloaded into the data directory following the instructions on GitHub. 

## Install packages and configure environment

In [2]:
%load_ext autoreload 
%autoreload 2

# Import external libraries
import dask
from datetime import datetime
from IPython.core.display import HTML
import numpy as np
import pyvista as pv
from dask.distributed import Client

# Import core python libraries
import os
import psutil

# Import local scripts
import brainz
import pipeline
import plotting
import preprocess
import util

# Configure environment - using threads because pipeline is data transfer heavy.
# Note : on windows and mac, docker desktop imposes virtual machine RAM limits
# by default. If the amount of memory seems unusually low, you can follow the
# instructions to increase the VM memory under Settings -> Resources, or you
# can try to be patient. Ideally this would be ran on a Linux.
mem = psutil.virtual_memory().total # Available RAM in bytes
mem_gb = mem / (1024**3) # Available RAM in GB
num_cores = os.cpu_count()
client = Client(processes=False, 
                threads_per_worker=num_cores, 
                memory_limit=0.5*mem)
print(f"{mem_gb} GB available on machine; using {0.5*mem_gb} GB")
print(f"Using {num_cores} threads")
dask.config.set({"dataframe.shuffle.method": "p2p"})
print(f"\nDask Dashboard at {client.dashboard_link}\n")
preprocess.run() # ~10 mins first run
pv.set_jupyter_backend("client")
HTML("""
<style>
.output_svg {
    display: table-cell;
    text-align: center;
    vertical-align: middle;
}
</style>
""")

# Set globals
OUTDIR = os.path.join(os.path.dirname(__name__), "results", "notebook")
if not os.path.exists(OUTDIR): os.mkdir(OUTDIR)
PERFDIR = os.path.join(os.path.dirname(__name__), "results", "performance_tests")
MINSIZE = 30 # Minimum number of nodes in a cluster/community

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
16969424896 available on machine; using 8484712448.0
Using 8 threads

Dask Dashboard at http://192.168.0.209:8787/status

Notice: this may take about 10 minutes if this is the first time running preprocess.py. 

12:23:25 Preprocessing complete!


2026-05-28 12:25:59,481 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 4a31ffea0e673623abeabb1fc11e0c4e initialized by task ('shuffle-transfer-4a31ffea0e673623abeabb1fc11e0c4e', 0) executed on worker inproc://192.168.0.209/24912/4
2026-05-28 12:25:59,584 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 4a31ffea0e673623abeabb1fc11e0c4e deactivated due to stimulus 'task-finished-1779927959.5828738'
2026-05-28 12:26:02,908 - distributed.worker.memory - WARNING - Unmanaged memory use is high. This may indicate a memory leak or the memory may not be released to the OS; see https://distributed.dask.org/en/latest/worker-memory.html#memory-not-released-back-to-the-os for more information. -- Unmanaged memory: 5.64 GiB -- Worker memory limit: 7.90 GiB
2026-05-28 12:26:04,007 - distributed.worker.memory - WARNING - Worker is at 81% memory usage. Pausing worker.  Process memory: 6.47 GiB -- Worker memory limit: 7.90 GiB
2026-05-28 12:26:04,150 - distributed.worker.memory -

## Preview - Can Nodule Neurons be Isolated from Linker Neurons?

Here I visualise the initial output generated from HDBSCAN clustering on xyz coordinates on a small sample of the drosophila connectome. The aim is to ensure the clustering parameters assign cluster IDs to groups in a way that approximates real nodules. As the 3D plot shows, the Drosophila connectome cannot be segregated into 'nodules' using this approach. Indeed, most synapses are equidistant from each other in 3D space. To isolate 'nodule'-like structures and generate visualisations such as those seen in the literature, filtering by cell type is required, however, cell type annotations are not present in this dataset. Interestingly, there do appear to be some patches of yellow, blue, and pink, indicating there may still be some differentiation in neurotransmitter probabilities across neuropils. The remainder of this analysis will focus on comparing neurotransmitter probabilities across neuropils using the unclustered dataframe.

In [ ]:
connectome = pipeline.load_connectome("data/tiny.parquet")
connectome = pipeline.normalise_nt_probs(connectome)
connectome = pipeline.attach_synapse_coords(connectome)
condensed = pipeline.condense(connectome)
condensed = util.do_hdbscan(condensed, MINSIZE)
clustered = pipeline.extend(condensed, connectome)

12:25:59 Loading connectome ...
12:25:59 Connectome loaded
12:25:59 Normalising neurotransmitter probabilities ...
12:25:59 Neurotransmitter probabilities normalised
12:25:59 Attaching coordinates ...


In [ ]:
# Plot 500,000 points
plotter = brainz.get_plotter(clustered, "hdbscan_id")
brainz.save(plotter, OUTDIR, _id="clusterd_brain_map")
print("Saved brain map!")
plotter.show()

In [ ]:
plotter.close() # Free memory

## Visualise Neurotransmitter Probability Distributions

### Overall Neurotransmitter Probability Distributions

In [ ]:
# Sample ~1 million points to speed up render (using matplotlib)
#sample = util.downsample(connectome, 1_000_000)
sample = connectome
filename = os.path.join(OUTDIR, "overall_distribution.svg")
plotting.plot_overall_distribution(sample, "Full Dataset", filename)

### Get Neuropil Summary Statistics

In [ ]:
neuropils = pipeline.get_neuropil_summary_stats(connectome)
neuropils.head(10)

### Neuropil Probability Distributions with respect to Size (Number of Synapses)

In [ ]:
filename = os.path.join(OUTDIR, "mean_neuropil_probs.svg")
plotting.plot_mean_nt_probs_by_neuropil_size(neuropils, filename)

### Neurotransmitter Probability Variance by Neuropil Size

In [ ]:
filename = os.path.join(OUTDIR, "neuropil_nt_prob_variance_by_size.svg")
plotting.plot_variance_by_neuropil_size(neuropils, filename)

### Neuropil Probability Distributions

In [ ]:
filename = os.path.join(OUTDIR, "neuropil_probability_distributions.svg")
plotting.plot_hex_per_neuropil(connectome, filename)

## Statistical Analysis

I want to know whether neurotransmitter probabilities are different between neuropils. Because probabilities are a form of compositional data (the sum of the variables is 1, and each variable is bounded between 0 and 1), the Dirichlet distribution is suitable. This statistical analysis is a simple one based on average synapse probabilities within neuropils. A more statistically robust method would involve using the 'other' column to calculate the ranges of excitatory and inhibitory probabilities for each synaptic connection, then running a statistical analysis that can handle comparing range values within a bounded interval, e.g., Bayesian Dirichlet regression with interval priors. The Dirichlet function I used can only operate on points, not ranges. There does not appear to be an out-of-box Dirichlet regression function in any Python libraries, so I will interface with the R DirichletReg package via rpy2.

In [ ]:
fitted, model_summary, null_summary, anova_result = do_stats.run(connectome)